# Stage 9a ablations — single-fold matrix with seed variance + reproducibility check (Kaggle T4)

Three reviewer-grade ablations on **fold 0 only** (Stage 9a's hardest fold at 0.5681; best signal-to-noise for ablations):
1. **lambda_ctc sweep** ∈ {0.0, 0.1, 0.3, 0.5, 0.7} — including the pure-attention endpoint.
2. **Decoder depth** ∈ {2, 3, 4} — around the headline 3-layer.
3. **Longer training** at 120 epochs.

Plus a **seed-variance estimate**: `l03` re-run with three different seeds (42, 43, 44) to bound how much of any ablation delta is genuine signal vs run-to-run noise.

And a **reproducibility check**: `l03_s0` uses the original Stage 9a seed (42) — its result should land within ±0.01 of the headline fold-0 number (0.5681) or the codebase has drifted.

| Variant | Changes | Epochs | lambda_ctc | Dec layers | Seed |
|---|---|---|---|---|---|
| `stage9a_abl_l03_s0` | reproducibility + control | 80 | 0.3 | 3 | **42** (original) |
| `stage9a_abl_l03_s1` | seed variance | 80 | 0.3 | 3 | **43** |
| `stage9a_abl_l03_s2` | seed variance | 80 | 0.3 | 3 | **44** |
| `stage9a_abl_e120`   | longer training | **120** | 0.3 | 3 | 42 |
| `stage9a_abl_l00`    | pure attention (no CTC) | 80 | **0.0** | 3 | 42 |
| `stage9a_abl_l01`    | more attention weight | 80 | **0.1** | 3 | 42 |
| `stage9a_abl_l05`    | balanced | 80 | **0.5** | 3 | 42 |
| `stage9a_abl_l07`    | more CTC weight | 80 | **0.7** | 3 | 42 |
| `stage9a_abl_d2`     | shallower decoder | 80 | 0.3 | **2** | 42 |
| `stage9a_abl_d4`     | deeper decoder | 80 | 0.3 | **4** | 42 |

**Wall-clock**: ~8 h 45 m on Kaggle T4.  Fits a single 9 h session **with margin**, but tighter than before — keep the kernel attached.

**Run order** (cheap-first per row of the matrix above is RE-ordered for execution): reproducibility check first; if `l03_s0` is outside ±0.02 of 0.5681, Cell 5 prints a warning and the rest of the sweep still runs (you can re-evaluate per-variant deltas after the fact).

**Prerequisites**: attach a notebook-output dataset that provides `skeleton_features_t32.pt` + `subject_cv5.json`.  Headline Stage 9a checkpoints are NOT required — every ablation trains from scratch.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate cache + manifest

In [ ]:
import os, glob, json, shutil
def _first(pattern):
    m = (glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True))
    return m[0] if m else None

CACHE_PATH        = _first('skeleton_features_t32.pt')
CV_MANIFEST_FOUND = _first('subject_cv5.json')
RESULTS_FOUND     = _first('stage9a_ablations_results.json')

OUT_CACHE    = CACHE_PATH        or '/kaggle/working/skeleton_features_t32.pt'
OUT_MANIFEST = CV_MANIFEST_FOUND or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage9a_ablations_results.json'
if RESULTS_FOUND and not os.path.exists(RESULTS_PATH):
    shutil.copy(RESULTS_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)

print(f'cache    : {OUT_CACHE}  (exists={os.path.exists(OUT_CACHE)})')
print(f'manifest : {OUT_MANIFEST}  (exists={os.path.exists(OUT_MANIFEST)})')
print(f'results  : {RESULTS_PATH}  (exists={os.path.exists(RESULTS_PATH)})')
assert os.path.exists(OUT_CACHE),    'Attach a dataset with skeleton_features_t32.pt'
assert os.path.exists(OUT_MANIFEST), 'Attach a dataset with subject_cv5.json'

## Cell 3 — Config + ablation matrix

In [ ]:
import logging, random
import numpy as np
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage9a_ablations.log'))])

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# Locked from Stage 9a.
T_NATIVE     = 32
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 32
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
WARMUP_PCT   = 0.05
DEC_N_HEADS  = 4
ABLATION_FOLD = 0

# (variant_name, num_epochs, lambda_ctc, dec_n_layers, seed)
# Execution order: reproducibility/seed-variance first, then headline-changing
# rows.  e120 last — it's the longest single run, slot it near the end so the
# earlier (faster) rows are always preserved if the session times out.
ABLATIONS = [
    ('stage9a_abl_l03_s0',  80, 0.3, 3, 42),   # reproducibility + control
    ('stage9a_abl_l03_s1',  80, 0.3, 3, 43),   # seed variance
    ('stage9a_abl_l03_s2',  80, 0.3, 3, 44),   # seed variance
    ('stage9a_abl_d2',      80, 0.3, 2, 42),
    ('stage9a_abl_d4',      80, 0.3, 4, 42),
    ('stage9a_abl_l00',     80, 0.0, 3, 42),   # pure-attention endpoint
    ('stage9a_abl_l01',     80, 0.1, 3, 42),
    ('stage9a_abl_l05',     80, 0.5, 3, 42),
    ('stage9a_abl_l07',     80, 0.7, 3, 42),
    ('stage9a_abl_e120',   120, 0.3, 3, 42),   # longest, scheduled last
]

# Headline reference for the reproducibility check (Stage 9a fold 0).
HEADLINE_FOLD0_CER = 0.5681
REPRO_TOLERANCE    = 0.02      # cudnn nondeterminism floor

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=42),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=80, batch_size=BATCH_SIZE, lr=LR_PEAK,
                      weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                      num_workers=2, warmup_pct=WARMUP_PCT, seed=42,
                      checkpoint_dir=CKPT_DIR),
).build()
torch.backends.cudnn.benchmark = True
print(f'Device         : {cfg.device}')
print(f'Ablation fold  : {ABLATION_FOLD}')
print(f'Variants       : {len(ABLATIONS)}')
print(f'Headline ref   : {HEADLINE_FOLD0_CER}  (tolerance ±{REPRO_TOLERANCE})')

## Cell 4 — Run the ablation matrix

Resume-aware: re-running this cell after a disconnect skips completed (fold, variant) pairs.  After `stage9a_abl_l03_s0` lands, prints a reproducibility verdict against the headline fold-0 number.

In [ ]:
from wita_v2.training.stage9_train import train_one_fold
from wita_v2.datasets.cv_splits     import fold_indices, load_cv5_manifest
from wita_v2.datasets.skeleton_augment import LandmarkAugment

cache    = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
manifest = load_cv5_manifest(OUT_MANIFEST)

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} ablations already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()
train_idx, val_idx = fold_indices(manifest, ABLATION_FOLD, cache['subjects'])
print(f'fold {ABLATION_FOLD}: train_clips={len(train_idx)}  val_clips={len(val_idx)}\n')

for variant_name, num_epochs, lambda_ctc, dec_n_layers, seed in ABLATIONS:
    if (ABLATION_FOLD, variant_name) in completed:
        print(f'[skip] {variant_name} (already done)'); continue
    print(f'\n>>> {variant_name}: epochs={num_epochs} '
          f'lambda={lambda_ctc} dec_l={dec_n_layers} seed={seed}')
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=ABLATION_FOLD, variant=variant_name,
        num_epochs=num_epochs, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
        dec_n_layers=dec_n_layers, dec_n_heads=DEC_N_HEADS,
        lambda_ctc=lambda_ctc, transform=train_aug, seed=seed,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    summary['_abl_num_epochs']   = num_epochs
    summary['_abl_lambda_ctc']   = lambda_ctc
    summary['_abl_dec_n_layers'] = dec_n_layers
    summary['_abl_seed']         = seed
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  best CER={result["best_val_cer"]:.4f} '
          f'(ep {result["best_epoch"]})  '
          f'len_ratio={result["best_mean_pred_len_ratio"]:.3f}')

    # Reproducibility verdict immediately after l03_s0.
    if variant_name == 'stage9a_abl_l03_s0':
        delta = abs(result['best_val_cer'] - HEADLINE_FOLD0_CER)
        if delta <= REPRO_TOLERANCE:
            print(f'  ✅ REPRO OK: |{result["best_val_cer"]:.4f} '
                  f'- {HEADLINE_FOLD0_CER}| = {delta:.4f} ≤ {REPRO_TOLERANCE}')
        else:
            print(f'  ⚠️  REPRO DRIFT: |{result["best_val_cer"]:.4f} '
                  f'- {HEADLINE_FOLD0_CER}| = {delta:.4f} > {REPRO_TOLERANCE}')
            print('     Codebase may have changed since the headline run.')
            print('     The sweep will continue; treat deltas as suggestive,')
            print('     not authoritative, until the drift source is identified.')

print(f'\n{len(all_results)}/{len(ABLATIONS)} ablations done.')

## Cell 5 — Seed-variance estimate (sets the 2σ threshold for claim-worthy deltas)

In [ ]:
import numpy as np
with open(RESULTS_PATH) as f:
    all_results = json.load(f)
by_variant = {r['variant']: r for r in all_results}

seed_runs = [by_variant.get(v) for v in
             ('stage9a_abl_l03_s0', 'stage9a_abl_l03_s1', 'stage9a_abl_l03_s2')]
seed_runs = [r for r in seed_runs if r is not None]
cers = [r['best_val_cer'] for r in seed_runs]
print(f'l03 seed-variance estimate (n={len(seed_runs)}):')
for r in seed_runs:
    print(f'  seed={r.get("_abl_seed", r.get("seed", "?"))}  best_val_cer={r["best_val_cer"]:.4f}')
if len(cers) >= 2:
    mu  = float(np.mean(cers))
    sig = float(np.std(cers, ddof=1)) if len(cers) > 1 else 0.0
    print(f'\n  mean = {mu:.4f}   std = {sig:.4f}')
    THRESHOLD_2SIG = max(0.005, 2.0 * sig)   # never claim a < 0.005 delta as real
    print(f'  2σ claim-worthy threshold for ablation deltas: {THRESHOLD_2SIG:.4f}')
else:
    THRESHOLD_2SIG = 0.02
    print(f'\n  Only one seed yet — falling back to a conservative threshold: {THRESHOLD_2SIG:.4f}')

## Cell 6 — Ablation tables with significance flags

In [ ]:
control = by_variant.get('stage9a_abl_l03_s0')
if control is None:
    raise RuntimeError('l03_s0 not yet run — cannot evaluate deltas.')
ctl_cer = control['best_val_cer']

def _flag(v_cer):
    d = v_cer - ctl_cer
    tag = '✅' if d <= -THRESHOLD_2SIG else ('❌' if d >= THRESHOLD_2SIG else '⚖')
    return d, tag

print(f'Control (l03_s0): {ctl_cer:.4f}  -  2σ threshold: {THRESHOLD_2SIG:.4f}\n')

print('=== lambda_ctc sweep (decoder=3 layers, 80 epochs, seed=42) ===')
print(' lambda    CER      Δ vs control    flag    best ep    len_ratio')
for v, lbl in [('stage9a_abl_l00', '0.0'), ('stage9a_abl_l01', '0.1'),
               ('stage9a_abl_l03_s0', '0.3'), ('stage9a_abl_l05', '0.5'),
               ('stage9a_abl_l07', '0.7')]:
    r = by_variant.get(v)
    if r is None:
        print(f'  {lbl:<7}  pending'); continue
    d, tag = _flag(r['best_val_cer'])
    print(f'  {lbl:<7}  {r["best_val_cer"]:.4f}     {d:+.4f}      {tag}      '
          f'{r["best_epoch"]:>3}     {r.get("best_mean_pred_len_ratio", float("nan")):.3f}')

print('\n=== Decoder depth (lambda=0.3, 80 epochs, seed=42) ===')
print(' depth    CER      Δ vs control    flag    best ep')
for v, lbl in [('stage9a_abl_d2', '2'),
               ('stage9a_abl_l03_s0', '3 (control)'),
               ('stage9a_abl_d4', '4')]:
    r = by_variant.get(v)
    if r is None: print(f'  {lbl:<13} pending'); continue
    d, tag = _flag(r['best_val_cer'])
    print(f'  {lbl:<13}  {r["best_val_cer"]:.4f}     {d:+.4f}      {tag}      {r["best_epoch"]}')

print('\n=== Training length (lambda=0.3, decoder=3, seed=42) ===')
for v, lbl in [('stage9a_abl_l03_s0', '80 ep (control)'),
               ('stage9a_abl_e120', '120 ep')]:
    r = by_variant.get(v)
    if r is None: print(f'  {lbl}  pending'); continue
    d, tag = _flag(r['best_val_cer'])
    print(f'  {lbl:<18}  CER {r["best_val_cer"]:.4f}  Δ {d:+.4f}  {tag}  '
          f'(best ep {r["best_epoch"]})')

print(f'\nFlags: ✅ improvement ≥ 2σ   ⚖ within seed noise   ❌ regression ≥ 2σ')

## Cell 7 — lambda sweep curve + decoder depth + 5 seed runs scatter

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))

# (1) lambda curve
lam, cer = [], []
for v, l in [('stage9a_abl_l00', 0.0), ('stage9a_abl_l01', 0.1),
             ('stage9a_abl_l03_s0', 0.3), ('stage9a_abl_l05', 0.5),
             ('stage9a_abl_l07', 0.7)]:
    if v in by_variant:
        lam.append(l); cer.append(by_variant[v]['best_val_cer'])
ax = axes[0]
ax.plot(lam, cer, 'o-', color='#1f77b4', markersize=8)
ax.axhline(ctl_cer, color='gray', linestyle=':', alpha=0.5, label='control')
ax.fill_between([min(lam, default=0), max(lam, default=1)],
                ctl_cer - THRESHOLD_2SIG, ctl_cer + THRESHOLD_2SIG,
                color='gray', alpha=0.15, label='±2σ seed band')
for x, y in zip(lam, cer):
    ax.annotate(f'{y:.3f}', (x, y), fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('lambda_ctc'); ax.set_ylabel('fold 0 best val CER')
ax.set_title('lambda_ctc sweep')
ax.set_xticks([0.0, 0.1, 0.3, 0.5, 0.7])
ax.legend(frameon=False, fontsize=7); ax.grid(True, linestyle=':', alpha=0.4)

# (2) depth curve
depth, cer_d = [], []
for v, d in [('stage9a_abl_d2', 2), ('stage9a_abl_l03_s0', 3), ('stage9a_abl_d4', 4)]:
    if v in by_variant:
        depth.append(d); cer_d.append(by_variant[v]['best_val_cer'])
ax = axes[1]
ax.plot(depth, cer_d, 's-', color='#d62728', markersize=8)
ax.axhline(ctl_cer, color='gray', linestyle=':', alpha=0.5)
ax.fill_between([min(depth, default=1), max(depth, default=4)],
                ctl_cer - THRESHOLD_2SIG, ctl_cer + THRESHOLD_2SIG,
                color='gray', alpha=0.15)
for x, y in zip(depth, cer_d):
    ax.annotate(f'{y:.3f}', (x, y), fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('decoder layers'); ax.set_ylabel('fold 0 best val CER')
ax.set_title('decoder depth'); ax.set_xticks([2, 3, 4])
ax.grid(True, linestyle=':', alpha=0.4)

# (3) seed scatter
seeds, cers = [], []
for v in ('stage9a_abl_l03_s0', 'stage9a_abl_l03_s1', 'stage9a_abl_l03_s2'):
    if v in by_variant:
        seeds.append(by_variant[v].get('_abl_seed', by_variant[v].get('seed', 0)))
        cers.append(by_variant[v]['best_val_cer'])
ax = axes[2]
ax.scatter(seeds, cers, s=80, c='#2ca02c')
for s, c in zip(seeds, cers):
    ax.annotate(f'{c:.4f}', (s, c), fontsize=7, xytext=(5, 0), textcoords='offset points')
ax.axhline(0.5681, color='black', linestyle='--', alpha=0.5, label='Stage 9a headline (0.5681)')
ax.set_xlabel('seed'); ax.set_ylabel('fold 0 best val CER')
ax.set_title('seed-variance probe (l03)')
ax.legend(frameon=False, fontsize=7); ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage9a_ablations_curves.png'), dpi=140)
plt.show()

## Cell 8 — Decisions to make from the table

Use the 2σ threshold from Cell 5; anything inside ±2σ is within seed noise and not claim-worthy.

**Re-run the full 5-fold headline ONLY when an ablation beats control by ≥ 2σ**:
1. `e120` better → re-run 5-fold at 120 epochs (~6 h).
2. `l05` or `l07` better → re-run 5-fold at that lambda.
3. `d4` better (or `d2` clearly better) → re-run 5-fold at that depth.

**Specific insights from the new λ=0.0 row**:
- `l00 ≈ l01` → CTC is largely an alignment regulariser at small weights; the report's claim about CTC's role should be softened.
- `l00 ≫ l01` (worse without CTC) → CTC is meaningfully contributing decoder signal even at λ=0.1.
- `l00` is the new headline → pure attention is the right design; drop CTC entirely (would simplify Stage 9b but lose the LM-fusion path).

**If `l03_s0` failed the reproducibility check** (warning in Cell 4): the ablation deltas are still internally consistent (paired against the same control), but the comparison to the original Stage 9a headline number is suspect.  Document this in the appendix; reviewers will want the explanation.

## Cell 9 — Commit kernel

In [ ]:
print('Save Version → Save & Run All to commit:')
print(f'  - {RESULTS_PATH}')
print(f'  - {LOG_DIR}/stage9a_ablations.log')
print(f'  - {LOG_DIR}/stage9a_ablations_curves.png')
print(f'  - {CKPT_DIR}/stage9a_fold0_stage9a_abl_*_best.pt  (10 checkpoints)')